In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Bidirectional, LSTM, Dense, Dropout,
    Lambda, Subtract, Multiply, concatenate, Dot
)
from tensorflow.keras.losses import Huber
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import mean_squared_error, mean_absolute_error
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── Environment Setup ────────────────────────────────────────────────────────────────────────────────
# Kaggle  : set USE_AUGMENTED = True/False sesuai kebutuhan
# Lokal   : ubah DATA_DIR ke path lokal

DATASET_SLUG  = "siamese-data"   # <-- GANTI sesuai nama dataset Kaggle
DATA_DIR      = f"/kaggle/input/{DATASET_SLUG}"
OUT_DIR       = "/kaggle/working"
USE_AUGMENTED = True             # True  → pakai aug_*.npy + aug_metadata.pkl
                                 # False → pakai data asli

gpus = tf.config.list_physical_devices('GPU')
print(f"TensorFlow  : {tf.__version__}")
print(f"GPU tersedia: {len(gpus)}")
for g in gpus:
    print(" ", g)

prefix = 'final_' if USE_AUGMENTED else ''
meta_f = 'final_metadata.pkl' if USE_AUGMENTED else 'metadata.pkl'
for fname in [f'{prefix}questions_emb.npy', f'{prefix}answerkeys_emb.npy',
              f'{prefix}answers_emb.npy',    meta_f]:
    path   = os.path.join(DATA_DIR, fname)
    status = "OK" if os.path.exists(path) else "TIDAK DITEMUKAN"
    print(f"  {fname:<30} -> {status}")

In [ ]:
# ── Load Data ────────────────────────────────────────────────────────────────────────────────────
# final_questions_emb & final_answerkeys_emb disimpan KOMPAK (1 per IDPSJ).
# Rekonstruksi array penuh menggunakan kolom psj_idx di metadata.

if USE_AUGMENTED:
    answers_emb    = np.load(os.path.join(DATA_DIR, 'final_answers_emb.npy'))
    uniq_q_emb     = np.load(os.path.join(DATA_DIR, 'final_questions_emb.npy'))
    uniq_ak_emb    = np.load(os.path.join(DATA_DIR, 'final_answerkeys_emb.npy'))
    metadata       = pd.read_pickle(os.path.join(DATA_DIR, 'final_metadata.pkl'))
else:
    answers_emb    = np.load(os.path.join(DATA_DIR, 'answers_emb.npy'))
    uniq_q_emb     = np.load(os.path.join(DATA_DIR, 'questions_emb.npy'))
    uniq_ak_emb    = np.load(os.path.join(DATA_DIR, 'answerkeys_emb.npy'))
    metadata       = pd.read_pickle(os.path.join(DATA_DIR, 'metadata.pkl'))

metadata = metadata.reset_index(drop=True)

# Jika data asli (belum kompak), psj_idx belum ada → buat sekarang
if 'psj_idx' not in metadata.columns:
    idpsj_sorted = sorted(metadata['IDPSJ'].unique())
    idpsj_to_idx = {psj: i for i, psj in enumerate(idpsj_sorted)}
    metadata['psj_idx'] = metadata['IDPSJ'].map(idpsj_to_idx)

# Rekonstruksi array penuh menggunakan psj_idx
questions_emb  = uniq_q_emb[metadata['psj_idx'].values]
answerkeys_emb = uniq_ak_emb[metadata['psj_idx'].values]

print("=== Hasil Load ===")
print(f"answers_emb    : {answers_emb.shape}    (per sampel)")
print(f"uniq_q_emb     : {uniq_q_emb.shape}   (kompak, per IDPSJ)")
print(f"questions_emb  : {questions_emb.shape}  (rekonstruksi)")
print(f"answerkeys_emb : {answerkeys_emb.shape} (rekonstruksi)")
print(f"\nMetadata       : {len(metadata)} rows")
print(f"Kolom metadata : {list(metadata.columns)}")
print(f"\nIDPSJ unik     : {sorted(metadata['IDPSJ'].unique())}")
print(f"\nDistribusi grade:")
print(metadata['grade'].value_counts().sort_index())

In [ ]:
# ── Diagnostik: Answerkey/Question Similarity vs Grade Mean Correlation ────────
# Jalankan cell ini SEBELUM training untuk menentukan apakah
# answerkey NN calibration (v11) layak dilakukan.
#
# Pertanyaan: apakah IDPSJ yang answerkey-nya mirip juga memiliki grade mean yang mirip?
# Jika korelasinya tinggi → answerkey NN calibration akan efektif.

import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

# ── 1. Hitung centroid per IDPSJ (masked mean pool) ───────────────────────────────────
def masked_mean_pool_np(emb):
    """Mean pool token embeddings, ignoring zero-padded tokens. emb: (seq, dim)"""
    mask = (np.abs(emb).sum(axis=-1) > 1e-6).astype(np.float32)   # (seq,)
    n = mask.sum()
    if n == 0:
        return emb.mean(axis=0)
    return (emb * mask[:, None]).sum(axis=0) / n

idpsj_sorted = sorted(metadata['IDPSJ'].unique())
n_idpsj = len(idpsj_sorted)
psj_to_idx = {p: i for i, p in enumerate(idpsj_sorted)}

# Centroid answerkey dan question per IDPSJ (dari compact arrays)
ak_centroids = np.stack([masked_mean_pool_np(uniq_ak_emb[i]) for i in range(n_idpsj)])  # (17, 300)
q_centroids  = np.stack([masked_mean_pool_np(uniq_q_emb[i])  for i in range(n_idpsj)])  # (17, 300)

ak_norm = ak_centroids / (np.linalg.norm(ak_centroids, axis=-1, keepdims=True) + 1e-8)
q_norm  = q_centroids  / (np.linalg.norm(q_centroids,  axis=-1, keepdims=True) + 1e-8)

# ── 2. Hitung pairwise cosine similarity antar IDPSJ ─────────────────────────────
ak_sim = ak_norm @ ak_norm.T   # (17, 17)
q_sim  = q_norm  @ q_norm.T    # (17, 17)

# ── 3. Hitung actual mean grade per IDPSJ (real samples only) ────────────────────
if 'is_synthetic' in metadata.columns:
    is_real_diag = ~metadata['is_synthetic'].values
else:
    is_real_diag = ~metadata['IDJwb'].astype(str).str.startswith(('syn_', 'cpi_')).values

real_meta = metadata[is_real_diag].copy()
mean_grade = np.array([
    real_meta[real_meta['IDPSJ'] == p]['grade'].mean()
    for p in idpsj_sorted
])  # (17,)

# Pairwise absolute difference in mean grade
grade_diff = np.abs(mean_grade[:, None] - mean_grade[None, :])  # (17, 17)

# ── 4. Flatten upper triangle (pairwise, exclude diagonal) ───────────────────────
triu_idx = np.triu_indices(n_idpsj, k=1)
ak_sim_flat    = ak_sim[triu_idx]
q_sim_flat     = q_sim[triu_idx]
grade_diff_flat = grade_diff[triu_idx]

# ── 5. Korelasi: similarity vs grade proximity (−|diff|) ─────────────────────────
r_ak_p, p_ak_p = pearsonr(ak_sim_flat, -grade_diff_flat)
r_ak_s, p_ak_s = spearmanr(ak_sim_flat, -grade_diff_flat)
r_q_p,  p_q_p  = pearsonr(q_sim_flat,  -grade_diff_flat)
r_q_s,  p_q_s  = spearmanr(q_sim_flat,  -grade_diff_flat)

print("=" * 60)
print("Korelasi: Similarity vs Grade Proximity (−|Δgrade_mean|)")
print("=" * 60)
print(f"  Answerkey cosine sim — Pearson  r={r_ak_p:+.4f}  p={p_ak_p:.4f}")
print(f"  Answerkey cosine sim — Spearman r={r_ak_s:+.4f}  p={p_ak_s:.4f}")
print(f"  Question  cosine sim — Pearson  r={r_q_p:+.4f}  p={p_q_p:.4f}")
print(f"  Question  cosine sim — Spearman r={r_q_s:+.4f}  p={p_q_s:.4f}")
print()
print("Interpretasi:")
print("  |r| > 0.5 dan p < 0.05 → NN calibration layak dicoba")
print("  |r| < 0.3 → NN calibration tidak akan membantu")

# ── 6. Tabel: mean grade + 3 IDPSJ terdekat per answerkey ────────────────────────
print(f"\n{'IDPSJ':>6} {'act_\u03bc':>6} | {'3 AK-NN terdekat (IDPSJ: sim, act_\u03bc)':}")
print("-" * 70)
for i, pid in enumerate(idpsj_sorted):
    sims_i = ak_sim[i].copy()
    sims_i[i] = -1  # exclude self
    top3_idx = np.argsort(sims_i)[::-1][:3]
    nn_str = "  ".join([
        f"IDPSJ{idpsj_sorted[j]}(sim={ak_sim[i,j]:.3f}, \u03bc={mean_grade[j]:.2f})"
        for j in top3_idx
    ])
    print(f"{pid:>6} {mean_grade[i]:>6.2f} | {nn_str}")

# ── 7. Plot ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.scatter(ak_sim_flat, grade_diff_flat, alpha=0.5, s=20)
ax.set_xlabel('Answerkey cosine similarity')
ax.set_ylabel('|\u0394 mean grade|')
ax.set_title(f'AK Similarity vs Grade Diff\nPearson r={r_ak_p:.3f}, p={p_ak_p:.4f}')
z = np.polyfit(ak_sim_flat, grade_diff_flat, 1)
x_line = np.linspace(ak_sim_flat.min(), ak_sim_flat.max(), 100)
ax.plot(x_line, np.polyval(z, x_line), 'r--', linewidth=1.5)

ax2 = axes[1]
ax2.scatter(q_sim_flat, grade_diff_flat, alpha=0.5, s=20, color='green')
ax2.set_xlabel('Question cosine similarity')
ax2.set_ylabel('|\u0394 mean grade|')
ax2.set_title(f'Q Similarity vs Grade Diff\nPearson r={r_q_p:.3f}, p={p_q_p:.4f}')
z2 = np.polyfit(q_sim_flat, grade_diff_flat, 1)
x_line2 = np.linspace(q_sim_flat.min(), q_sim_flat.max(), 100)
ax2.plot(x_line2, np.polyval(z2, x_line2), 'r--', linewidth=1.5)

ax3 = axes[2]
for i, pid in enumerate(idpsj_sorted):
    ax3.scatter(ak_sim[i, i], mean_grade[i], s=0)
import matplotlib.colors as mcolors
im = ax3.imshow(grade_diff, cmap='RdYlGn_r', vmin=0, vmax=4)
ax3.set_xticks(range(n_idpsj)); ax3.set_xticklabels(idpsj_sorted, fontsize=7)
ax3.set_yticks(range(n_idpsj)); ax3.set_yticklabels(idpsj_sorted, fontsize=7)
ax3.set_title('Pairwise |\u0394 mean grade| (merah=jauh)')
plt.colorbar(im, ax=ax3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'diag_ak_sim_vs_grade.png'), dpi=150)
plt.show()

# ── 8. Simulasi: jika pakai AK NN calibration, berapa estimasi mean per fold? ─
print("\n" + "=" * 70)
print("Simulasi AK NN Calibration (berapa estimasi mean grade untuk test IDPSJ?)")
print("=" * 70)
print(f"{'IDPSJ':>6} {'act_\u03bc':>6} | {'Est. via AK-NN':>14} {'Error kalibrasi':>16} | {'Catatan':}")
print("-" * 75)

def estimate_mean_via_nn(test_idx, train_indices, ak_norm, mean_grade, temperature=10.0):
    sims = ak_norm[train_indices] @ ak_norm[test_idx]          # (n_train,)
    weights = np.exp(sims * temperature)
    weights /= weights.sum()
    return float(np.dot(weights, mean_grade[train_indices]))

all_errors = []
for fold_i, test_pid in enumerate(idpsj_sorted):
    val_pid    = idpsj_sorted[(fold_i + 1) % n_idpsj]
    train_pids = [p for p in idpsj_sorted if p != test_pid and p != val_pid]
    test_i     = psj_to_idx[test_pid]
    train_is   = [psj_to_idx[p] for p in train_pids]

    est = estimate_mean_via_nn(test_i, train_is, ak_norm, mean_grade)
    err = abs(est - mean_grade[test_i])
    all_errors.append(err)
    flag = "  \u2190 masalah" if err > 1.5 else ("  \u2190 ok" if err < 0.5 else "")
    print(f"{test_pid:>6} {mean_grade[test_i]:>6.2f} | {est:>14.2f} {err:>+15.3f} |{flag}")

print(f"\nRata-rata |error estimasi mean|: {np.mean(all_errors):.4f}")
print(f"(Bandingkan dengan rata-rata mean bias v10:  {np.mean([1.54,0.46,0.24,0.91,1.62,0.06,0.39,1.37,0.88,0.65,0.76,0.16,0.23,0.03,0.75,0.46,0.94]):.4f})")
print("\nJika |error estimasi mean| < |v10 mean bias| \u2192 AK-NN calibration layak.")

In [ ]:
# ── Hyperparameter (tuned via Optuna, best MAE=1.8615 on 4-fold proxy) ────────
BILSTM_UNITS  = 128
DROPOUT       = 0.40
EPOCHS        = 150
BATCH_SIZE    = 16
PATIENCE      = 15
LR            = 2.68e-3
N_RUNS        = 3

# ── Ordinal Regression Loss & Metric ──────────────────────────────────────────────────
def ordinal_loss(y_true, y_pred):
    """Sum of binary cross-entropies: apakah grade > k? (k=1..9)"""
    thresholds   = tf.cast(tf.range(1, 10), tf.float32)
    y_true_exp   = tf.expand_dims(tf.cast(y_true, tf.float32), -1)
    y_binary     = tf.cast(y_true_exp > thresholds, tf.float32)
    return tf.reduce_mean(
        tf.keras.losses.binary_crossentropy(y_binary, y_pred)
    )

def ordinal_mae(y_true, y_pred):
    """Grade prediction = jumlah threshold yang terlampaui + 1."""
    grade_pred = tf.reduce_sum(tf.cast(y_pred > 0.5, tf.float32), axis=-1) + 1.0
    return tf.reduce_mean(tf.abs(tf.cast(y_true, tf.float32) - grade_pred))


def attention_pool(seq_out, name_prefix):
    """Soft attention pooling atas output BiLSTM (return_sequences=True)."""
    score   = Dense(1, activation='tanh', use_bias=False,
                    name=f'{name_prefix}_attn_score')(seq_out)
    weights = Lambda(lambda x: tf.nn.softmax(x, axis=1),
                    name=f'{name_prefix}_attn_w')(score)
    pooled  = Lambda(lambda x: tf.reduce_sum(x[0] * x[1], axis=1),
                    name=f'{name_prefix}_attn_pool')([seq_out, weights])
    return pooled


def build_model(q_seq_len, ak_seq_len, a_seq_len, emb_dim=300,
                n_scalar=10,
                bilstm_units=BILSTM_UNITS, dropout=DROPOUT):
    """
    Siamese BiLSTM v11 — Ordinal Regression + Normalized + Centroid
                         + cos_sim_q_a + orisinalitas

    Perbedaan dari v10:
    - Tambahkan kembali cos_sim_q_a dan orisinalitas (= 1 - cos_sim_q_a)
      yang sebelumnya hilang saat migrasi ke arsitektur direct.
    - cos_sim_q_a  : cosine similarity antara repr. question dan answer (BiLSTM)
                     \u2192 mengukur relevansi jawaban terhadap pertanyaan
    - orisinalitas : 1 - cos_sim_q_a
                     \u2192 penalti jika jawaban hanya menyalin ulang pertanyaan
    - Sesuai tujuan penelitian: model mengevaluasi kemiripan jawaban terhadap
      kunci jawaban SEKALIGUS relevansinya terhadap pertanyaan.

    Merged: [ea(256), eak(256), eq(256), abs_diff(256), had_prod(256),
             cos_sim_ak_a(1), cos_sim_q_a(1), orisinalitas(1),
             scalar_dense(32)]  = 1315D
    Head: Dense(512) \u2192 Dropout \u2192 Dense(64) \u2192 Dense(9, sigmoid)  [ordinal]
    """
    shared_bilstm = Bidirectional(
        LSTM(bilstm_units, return_sequences=True), name='bilstm_shared'
    )

    inp_q      = Input(shape=(q_seq_len,  emb_dim), name='inp_q')
    inp_ak     = Input(shape=(ak_seq_len, emb_dim), name='inp_ak')
    inp_a      = Input(shape=(a_seq_len,  emb_dim), name='inp_a')
    inp_scalar = Input(shape=(n_scalar,),            name='inp_scalar')

    # ── BiLSTM + Attention Pooling ───────────────────────────────────────────────────
    eq_seq  = shared_bilstm(inp_q)
    eak_seq = shared_bilstm(inp_ak)
    ea_seq  = shared_bilstm(inp_a)

    eq  = attention_pool(eq_seq,  'q')
    eak = attention_pool(eak_seq, 'ak')
    ea  = attention_pool(ea_seq,  'a')

    # ── Kemiripan answerkey vs answer (task utama Siamese) ───────────────────────
    abs_diff     = Lambda(lambda x: tf.abs(x[0] - x[1]), name='abs_diff')([eak, ea])
    had_prod     = Multiply(name='had_prod')([eak, ea])
    cos_sim_ak_a = Dot(axes=1, normalize=True, name='cos_sim_ak_a')([eak, ea])

    # ── Relevansi jawaban terhadap pertanyaan (penalti menyalin soal) ────────────
    cos_sim_q_a  = Dot(axes=1, normalize=True, name='cos_sim_q_a')([eq, ea])
    orisinalitas = Lambda(lambda x: 1.0 - x, name='orisinalitas')(cos_sim_q_a)

    # ── Scalar branch: normalized + centroid ───────────────────────────────────────
    scalar_feat = Dense(32, activation='relu', name='scalar_dense')(inp_scalar)

    merged = concatenate(
        [ea, eak, eq, abs_diff, had_prod,
         cos_sim_ak_a, cos_sim_q_a, orisinalitas,
         scalar_feat],
        name='merged'
    )

    x   = Dense(512, activation='relu')(merged)
    x   = Dropout(dropout)(x)
    x   = Dense(64, activation='relu')(x)
    out = Dense(9, activation='sigmoid', name='ordinal_out')(x)

    model = Model(
        inputs=[inp_q, inp_ak, inp_a, inp_scalar],
        outputs=out,
        name='siamese_bilstm_v11'
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
        loss=ordinal_loss,
        metrics=[ordinal_mae]
    )
    return model


# Verifikasi arsitektur (n_scalar=10: 5 normalized + 5 centroid)
_tmp = build_model(
    q_seq_len  = questions_emb.shape[1],
    ak_seq_len = answerkeys_emb.shape[1],
    a_seq_len  = answers_emb.shape[1],
    emb_dim    = answers_emb.shape[2],
    n_scalar   = 10
)
_tmp.summary()
del _tmp

In [ ]:
# ── Hyperparameter Search (Optuna) ────────────────────────────────────────────────
# Jalankan cell ini sebelum LOPO training untuk menemukan hyperparameter terbaik.
# Menggunakan 4 fold (subset) dan N_RUNS=1 agar lebih cepat (~30 menit di GPU).
# Setelah selesai, perbarui konstanta di cell build_model dengan nilai terbaik.

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEARCH_FOLD_INDICES = [0, 3, 6, 9]   # 4 fold representatif dari 12
SEARCH_EPOCHS       = 80
SEARCH_PATIENCE     = 8
N_TRIALS            = 40

idpsj_list_search = sorted(metadata['IDPSJ'].unique())
y_all_s   = metadata['grade'].values.astype(np.float32)
is_real_s = ~metadata['IDJwb'].astype(str).str.startswith('syn_')


def run_one_fold(fold_i, bilstm_units, dropout, lr, batch_size, dense1, dense2):
    test_id   = idpsj_list_search[fold_i]
    val_id    = idpsj_list_search[(fold_i + 1) % len(idpsj_list_search)]
    train_ids = [p for p in idpsj_list_search if p != test_id and p != val_id]

    tr_idx  = metadata.index[metadata['IDPSJ'].isin(train_ids)].values
    val_idx = metadata.index[(metadata['IDPSJ'] == val_id)  & is_real_s].values
    te_idx  = metadata.index[(metadata['IDPSJ'] == test_id) & is_real_s].values

    def gs(arr, idx): return arr[idx].astype(np.float32)

    y_tr = y_all_s[tr_idx];  y_v = y_all_s[val_idx];  y_te = y_all_s[te_idx]

    grade_int = y_tr.astype(int)
    ug, uc    = np.unique(grade_int, return_counts=True)
    fm        = dict(zip(ug, uc))
    raw_w     = np.array([len(y_tr) / (len(ug) * fm[g]) for g in grade_int])
    sw        = raw_w / raw_w.mean()

    # Buat model dengan hyperparameter saat ini
    shared = Bidirectional(LSTM(bilstm_units, return_sequences=False), name='bilstm_shared')

    inp_q  = Input(shape=(questions_emb.shape[1],  answers_emb.shape[2]), name='inp_q')
    inp_ak = Input(shape=(answerkeys_emb.shape[1], answers_emb.shape[2]), name='inp_ak')
    inp_a  = Input(shape=(answers_emb.shape[1],    answers_emb.shape[2]), name='inp_a')

    raw_ak_m = Lambda(lambda x: tf.reduce_mean(x, axis=1), name='raw_ak_mean')(inp_ak)
    raw_a_m  = Lambda(lambda x: tf.reduce_mean(x, axis=1), name='raw_a_mean')(inp_a)
    raw_q_m  = Lambda(lambda x: tf.reduce_mean(x, axis=1), name='raw_q_mean')(inp_q)
    rcos_ak  = Dot(axes=1, normalize=True, name='raw_cos_ak_a')([raw_ak_m, raw_a_m])
    rcos_q   = Dot(axes=1, normalize=True, name='raw_cos_q_a')([raw_q_m, raw_a_m])

    a_l  = Lambda(lambda x: tf.reduce_sum(tf.cast(tf.reduce_any(tf.abs(x)>1e-6, axis=-1), tf.float32), axis=-1, keepdims=True), name='a_len')(inp_a)
    ak_l = Lambda(lambda x: tf.reduce_sum(tf.cast(tf.reduce_any(tf.abs(x)>1e-6, axis=-1), tf.float32), axis=-1, keepdims=True), name='ak_len')(inp_ak)
    rlr  = Lambda(lambda x: x[0]/(x[1]+1e-8), name='raw_len_ratio')([a_l, ak_l])

    eak = shared(inp_ak);  ea = shared(inp_a)
    abd = Lambda(lambda x: tf.abs(x[0]-x[1]), name='abs_diff')([eak, ea])
    had = Multiply(name='had_prod')([eak, ea])
    cos = Dot(axes=1, normalize=True, name='cos_sim_ak_a')([eak, ea])

    mg  = concatenate([ea, eak, abd, had, rcos_ak, rcos_q, cos, rlr], name='merged')
    x   = Dense(dense1, activation='relu')(mg)
    x   = Dropout(dropout)(x)
    x   = Dense(dense2, activation='relu')(x)
    out = Dense(1, activation='linear')(x)

    m = Model(inputs=[inp_q, inp_ak, inp_a], outputs=out)
    m.compile(optimizer=tf.keras.optimizers.Adam(lr),
              loss=tf.keras.losses.Huber(delta=1.0), metrics=['mae'])

    cb = [EarlyStopping(monitor='val_mae', patience=SEARCH_PATIENCE,
                        restore_best_weights=True, verbose=0),
          ReduceLROnPlateau(monitor='val_mae', factor=0.5, patience=3,
                            min_lr=1e-6, verbose=0)]
    m.fit([gs(questions_emb, tr_idx),  gs(answerkeys_emb, tr_idx),  gs(answers_emb, tr_idx)], y_tr,
          sample_weight=sw,
          validation_data=([gs(questions_emb, val_idx), gs(answerkeys_emb, val_idx), gs(answers_emb, val_idx)], y_v),
          epochs=SEARCH_EPOCHS, batch_size=batch_size, callbacks=cb, verbose=0)

    yp = np.clip(np.round(m.predict([gs(questions_emb, te_idx), gs(answerkeys_emb, te_idx), gs(answers_emb, te_idx)], verbose=0).flatten()), 1, 10)
    tf.keras.backend.clear_session()
    return mean_absolute_error(y_te, yp)


def objective(trial):
    bilstm_units = trial.suggest_categorical('bilstm_units', [64, 128, 192, 256])
    dropout      = trial.suggest_float('dropout', 0.1, 0.5, step=0.05)
    lr           = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
    batch_size   = trial.suggest_categorical('batch_size', [16, 32, 64])
    dense1       = trial.suggest_categorical('dense1', [128, 256, 512])
    dense2       = trial.suggest_categorical('dense2', [32, 64, 128])

    maes = []
    for fi in SEARCH_FOLD_INDICES:
        try:
            mae = run_one_fold(fi, bilstm_units, dropout, lr, batch_size, dense1, dense2)
            maes.append(mae)
        except Exception as e:
            print(f"  Trial gagal fold {fi}: {e}")
            return float('inf')
    mean_mae = np.mean(maes)
    print(f"  Trial {trial.number:3d} | bilstm={bilstm_units} drop={dropout:.2f} "
          f"lr={lr:.1e} bs={batch_size} d1={dense1} d2={dense2} \u2192 MAE={mean_mae:.4f}")
    return mean_mae


print(f"Memulai Optuna search: {N_TRIALS} trial \u00d7 {len(SEARCH_FOLD_INDICES)} fold")
study = optuna.create_study(direction='minimize',
                             sampler=optuna.samplers.TPESampler())
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

best = study.best_params
print(f"\n{'='*55}")
print(f"Hyperparameter Terbaik (MAE={study.best_value:.4f}):")
print(f"  BILSTM_UNITS = {best['bilstm_units']}")
print(f"  DROPOUT      = {best['dropout']}")
print(f"  LR           = {best['lr']:.2e}")
print(f"  BATCH_SIZE   = {best['batch_size']}")
print(f"  Dense(1)     = {best['dense1']}")
print(f"  Dense(2)     = {best['dense2']}")
print(f"{'='*55}")
print("\u2192 Perbarui konstanta di cell build_model dengan nilai di atas.")

# Top-5 trial
top5 = study.trials_dataframe().sort_values('value').head(5)[['number','value','params_bilstm_units','params_dropout','params_lr','params_batch_size','params_dense1','params_dense2']]
print("\nTop-5 trial:")
print(top5.to_string(index=False))

In [ ]:
# ── Helper: precompute scalar features (dilakukan 1x sebelum LOPO loop) ─────────
from sklearn.metrics import cohen_kappa_score

def compute_scalar_features_all(answers_emb, answerkeys_emb, verbose=True):
    n = answers_emb.shape[0]
    feats = np.zeros((n, 5), dtype=np.float32)
    if verbose:
        print(f"Precomputing scalar features untuk {n} sampel...")
    for i in range(n):
        if verbose and i % 500 == 0:
            print(f"  {i}/{n}")
        a  = answers_emb[i].astype(np.float64)
        ak = answerkeys_emb[i].astype(np.float64)

        ak_norm = ak / (np.linalg.norm(ak, axis=-1, keepdims=True) + 1e-8)
        a_norm  = a  / (np.linalg.norm(a,  axis=-1, keepdims=True) + 1e-8)

        ak_mask = np.abs(ak).sum(axis=-1) > 1e-6
        a_mask  = np.abs(a ).sum(axis=-1) > 1e-6
        ak_n    = ak_norm[ak_mask]
        a_n     = a_norm[a_mask]

        if ak_n.shape[0] == 0 or a_n.shape[0] == 0:
            continue

        sim = ak_n @ a_n.T
        rec = float(sim.max(axis=1).mean())
        pre = float(sim.max(axis=0).mean())
        f1  = 2.0 * rec * pre / (rec + pre + 1e-8)

        m_ak = ak_n.mean(axis=0)
        m_a  = a_n.mean(axis=0)
        cos  = float(m_ak @ m_a / (np.linalg.norm(m_ak) * np.linalg.norm(m_a) + 1e-8))
        lrat = float(a_mask.sum()) / max(float(ak_mask.sum()), 1.0)

        feats[i] = [rec, pre, f1, cos, lrat]

    if verbose:
        print(f"  Selesai. Shape: {feats.shape}")
    return feats


def make_scalar_input_train(feats, idpsj_ids):
    norm_part = np.zeros_like(feats)
    cent_part = np.zeros_like(feats)
    for idpsj in np.unique(idpsj_ids):
        m = idpsj_ids == idpsj
        mu = feats[m].mean(axis=0)
        sg = feats[m].std(axis=0) + 1e-8
        norm_part[m] = (feats[m] - mu) / sg
        cent_part[m] = mu
    norm_part = np.clip(norm_part, -3.0, 3.0)
    return np.hstack([norm_part, cent_part]).astype(np.float32)


def make_scalar_input_test(feats):
    mu = feats.mean(axis=0)
    sg = feats.std(axis=0) + 1e-8
    norm_part = np.clip((feats - mu) / sg, -3.0, 3.0)
    cent_part = np.tile(mu, (len(feats), 1))
    return np.hstack([norm_part, cent_part]).astype(np.float32)


def ordinal_predict(model, X_q, X_ak, X_a, X_scalar=None):
    inputs = [X_q, X_ak, X_a]
    if X_scalar is not None:
        inputs.append(X_scalar)
    sigmoid_out = model.predict(inputs, verbose=0)
    grade = np.sum(sigmoid_out > 0.5, axis=-1) + 1
    return grade.astype(np.float32), sigmoid_out


# ── Precompute scalar features (1x untuk semua sampel) ──────────────────────────
all_scalar_feats = compute_scalar_features_all(answers_emb, answerkeys_emb)
N_SCALAR = all_scalar_feats.shape[1] * 2   # 5 normalized + 5 centroid = 10

# ── LOPO Cross-Validation + Ensemble ─────────────────────────────────────────────
idpsj_list = sorted(metadata['IDPSJ'].unique())
n_parts    = len(idpsj_list)
y_all      = metadata['grade'].values.astype(np.float32)

if 'is_synthetic' in metadata.columns:
    is_real = ~metadata['is_synthetic'].values
else:
    is_real = ~metadata['IDJwb'].astype(str).str.startswith(('syn_', 'cpi_')).values

fold_results = []

for i, test_id in enumerate(idpsj_list):
    val_id    = idpsj_list[(i + 1) % n_parts]
    train_ids = [pid for pid in idpsj_list if pid != test_id and pid != val_id]

    print(f"\n{'='*60}")
    print(f"Fold {i+1:02d}/{n_parts}  |  Test={test_id}  |  Val={val_id}")

    train_idx = metadata.index[metadata['IDPSJ'].isin(train_ids)].values
    val_idx   = metadata.index[(metadata['IDPSJ'] == val_id)  & is_real].values
    test_idx  = metadata.index[(metadata['IDPSJ'] == test_id) & is_real].values

    y_train = y_all[train_idx]
    y_val   = y_all[val_idx]
    y_test  = y_all[test_idx]

    print(f"  Train: {len(y_train)}  Val: {len(y_val)}  Test: {len(y_test)}")

    def get_split(arr, idx):
        return arr[idx].astype(np.float32)

    X_q_tr   = get_split(questions_emb,  train_idx)
    X_ak_tr  = get_split(answerkeys_emb, train_idx)
    X_a_tr   = get_split(answers_emb,    train_idx)
    X_q_val  = get_split(questions_emb,  val_idx)
    X_ak_val = get_split(answerkeys_emb, val_idx)
    X_a_val  = get_split(answers_emb,    val_idx)
    X_q_te   = get_split(questions_emb,  test_idx)
    X_ak_te  = get_split(answerkeys_emb, test_idx)
    X_a_te   = get_split(answers_emb,    test_idx)

    train_idpsj_ids = metadata['IDPSJ'].values[train_idx]
    scalar_tr  = make_scalar_input_train(all_scalar_feats[train_idx], train_idpsj_ids)
    scalar_val = make_scalar_input_test(all_scalar_feats[val_idx])
    scalar_te  = make_scalar_input_test(all_scalar_feats[test_idx])

    grade_int          = y_train.astype(int)
    unique_g, counts_g = np.unique(grade_int, return_counts=True)
    freq_map           = dict(zip(unique_g, counts_g))
    n_kelas            = len(unique_g)
    raw_w              = np.array([len(y_train) / (n_kelas * freq_map[g]) for g in grade_int])
    sample_w           = raw_w / raw_w.mean()

    # ── Ensemble: N_RUNS run per fold (tanpa fixed seed) ───────────────────────────
    all_sigmoid_raw = []

    for run in range(N_RUNS):
        print(f"\n  -- Run {run+1}/{N_RUNS} --")

        model = build_model(
            q_seq_len    = questions_emb.shape[1],
            ak_seq_len   = answerkeys_emb.shape[1],
            a_seq_len    = answers_emb.shape[1],
            emb_dim      = answers_emb.shape[2],
            n_scalar     = N_SCALAR,
            bilstm_units = BILSTM_UNITS,
            dropout      = DROPOUT
        )

        reduce_lr  = ReduceLROnPlateau(monitor='val_ordinal_mae', factor=0.5,
                                       patience=3, min_lr=1e-6, mode='min', verbose=0)
        early_stop = EarlyStopping(monitor='val_ordinal_mae', patience=PATIENCE,
                                   restore_best_weights=True, mode='min', verbose=1)

        model.fit(
            [X_q_tr, X_ak_tr, X_a_tr, scalar_tr], y_train,
            sample_weight=sample_w,
            validation_data=([X_q_val, X_ak_val, X_a_val, scalar_val], y_val),
            epochs=EPOCHS, batch_size=BATCH_SIZE,
            callbacks=[reduce_lr, early_stop], verbose=1
        )

        grade_s, sigmoid_s = ordinal_predict(model, X_q_te, X_ak_te, X_a_te, scalar_te)
        all_sigmoid_raw.append(sigmoid_s)

        mae_s = mean_absolute_error(y_test, np.clip(grade_s, 1, 10))
        print(f"  Run {run+1} MAE: {mae_s:.4f}")

    # ── Rata-rata sigmoid ensemble \u2192 grade final ────────────────────────────────────
    mean_sigmoid = np.mean(all_sigmoid_raw, axis=0)
    y_pred_final = np.clip(
        np.round(np.sum(mean_sigmoid > 0.5, axis=-1) + 1), 1, 10
    ).astype(np.float32)

    mae_final  = mean_absolute_error(y_test, y_pred_final)
    rmse_final = np.sqrt(mean_squared_error(y_test, y_pred_final))
    qwk_final  = cohen_kappa_score(y_test.astype(int), y_pred_final.astype(int),
                                   weights='quadratic')

    print(f"\n  MAE: {mae_final:.4f}  |  RMSE: {rmse_final:.4f}  |  QWK: {qwk_final:.4f}")

    fold_results.append({
        'fold'      : i + 1,
        'test_idpsj': test_id,
        'val_idpsj' : val_id,
        'n_train'   : len(y_train),
        'n_val'     : len(y_val),
        'n_test'    : len(y_test),
        'mae'       : mae_final,
        'rmse'      : rmse_final,
        'qwk'       : qwk_final,
        'y_test'    : y_test,
        'y_pred'    : y_pred_final,
    })

    model_path = os.path.join(OUT_DIR, f'model_fold_{i+1:02d}.keras')
    model.save(model_path)
    print(f"  Model saved -> {model_path}")

    tf.keras.backend.clear_session()

print("\n\nSelesai semua fold.")

In [ ]:
# ── Evaluasi Akhir ──────────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

summary = pd.DataFrame([{
    'fold'      : r['fold'],
    'test_idpsj': r['test_idpsj'],
    'n_test'    : r['n_test'],
    'MAE'       : round(r['mae'],  4),
    'RMSE'      : round(r['rmse'], 4),
    'QWK'       : round(r['qwk'],  4),
} for r in fold_results])

print("=" * 65)
print("Hasil per Fold \u2014 Siamese BiLSTM v11 (Ordinal + Norm + Centroid)")
print("=" * 65)
print(summary.to_string(index=False))
print(f"\nMAE  : {summary['MAE'].mean():.4f}  \u00b1  {summary['MAE'].std():.4f}")
print(f"RMSE : {summary['RMSE'].mean():.4f}  \u00b1  {summary['RMSE'].std():.4f}")
print(f"QWK  : {summary['QWK'].mean():.4f}  \u00b1  {summary['QWK'].std():.4f}")

summary.to_csv(os.path.join(OUT_DIR, 'lopo_results_v11_no_seed.csv'), index=False)

y_all_true = np.concatenate([r['y_test'] for r in fold_results])
y_all_pred = np.concatenate([r['y_pred'] for r in fold_results])
x          = np.arange(len(summary))

# ── Gambar 1: MAE per Fold ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x, summary['MAE'], color='steelblue', alpha=0.85,
       edgecolor='black', linewidth=0.5)
ax.axhline(summary['MAE'].mean(), color='red', linestyle='--', linewidth=1.5,
           label=f"Mean MAE = {summary['MAE'].mean():.4f}")
ax.set_xticks(x)
ax.set_xticklabels(summary['test_idpsj'].astype(str))
ax.set_xlabel('Test IDPSJ')
ax.set_ylabel('MAE')
ax.set_title('MAE per Fold \u2014 Siamese BiLSTM v11 (No Seed)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'v11_no_seed_plot1_mae_per_fold.png'), dpi=150)
plt.show()

# ── Gambar 2: Scatter Plot Prediksi vs Aktual ──────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_all_true, y_all_pred, alpha=0.4, edgecolors='k', linewidths=0.3)
ax.plot([1, 10], [1, 10], 'r--', label='Ideal')
ax.set_xlabel('Grade Aktual')
ax.set_ylabel('Grade Prediksi')
ax.set_title(f'Prediksi vs Aktual \u2014 Siamese BiLSTM v11 (No Seed)\n'
             f'MAE={summary["MAE"].mean():.4f}  '
             f'RMSE={summary["RMSE"].mean():.4f}  '
             f'QWK={summary["QWK"].mean():.4f}')
ax.set_xticks(range(1, 11))
ax.set_yticks(range(1, 11))
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'v11_no_seed_plot2_scatter.png'), dpi=150)
plt.show()

# ── Gambar 3: RMSE per Fold ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x, summary['RMSE'], color='mediumseagreen', alpha=0.85,
       edgecolor='black', linewidth=0.5)
ax.axhline(summary['RMSE'].mean(), color='red', linestyle='--', linewidth=1.5,
           label=f"Mean RMSE = {summary['RMSE'].mean():.4f}")
ax.set_xticks(x)
ax.set_xticklabels(summary['test_idpsj'].astype(str))
ax.set_xlabel('Test IDPSJ')
ax.set_ylabel('RMSE')
ax.set_title('RMSE per Fold \u2014 Siamese BiLSTM v11 (No Seed)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'v11_no_seed_plot3_rmse_per_fold.png'), dpi=150)
plt.show()

# ── Gambar 4: QWK per Fold ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x, summary['QWK'], color='mediumpurple', alpha=0.85,
       edgecolor='black', linewidth=0.5)
ax.axhline(summary['QWK'].mean(), color='darkviolet', linestyle='--', linewidth=1.5,
           label=f"Mean QWK = {summary['QWK'].mean():.4f}")
ax.axhline(0.6, color='gray', linestyle=':', linewidth=1.2, label='Threshold 0.6 (substantial)')
ax.set_xticks(x)
ax.set_xticklabels(summary['test_idpsj'].astype(str))
ax.set_xlabel('Test IDPSJ')
ax.set_ylabel('QWK')
ax.set_ylim(0, 1)
ax.set_title('Quadratic Weighted Kappa per Fold \u2014 Siamese BiLSTM v11 (No Seed)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'v11_no_seed_plot4_qwk_per_fold.png'), dpi=150)
plt.show()

print("\nGambar disimpan:")
for name in ['v11_no_seed_plot1_mae_per_fold.png', 'v11_no_seed_plot2_scatter.png',
             'v11_no_seed_plot3_rmse_per_fold.png', 'v11_no_seed_plot4_qwk_per_fold.png']:
    print(f"  {os.path.join(OUT_DIR, name)}")

In [ ]:
# ── Diagnosis: Grade Distribution vs Prediksi per IDPSJ ────────────────────────────
import matplotlib.pyplot as plt

if 'is_synthetic' in metadata.columns:
    is_real_diag = ~metadata['is_synthetic'].values
else:
    is_real_diag = ~metadata['IDJwb'].astype(str).str.startswith(('syn_', 'cpi_')).values

real_meta = metadata[is_real_diag].copy()

# ── Tabel: statistik per IDPSJ ───────────────────────────────────────────────────────
print(f"{'IDPSJ':>6} {'n':>4} {'act_\u03bc':>6} {'act_\u03c3':>6} {'%\u22643':>6} {'%\u22657':>6} "
      f"| {'pred_\u03bc':>7} {'MAE':>8} {'RMSE':>8} {'QWK':>7}")
print("-" * 78)
for r in sorted(fold_results, key=lambda x: x['test_idpsj']):
    tid  = r['test_idpsj']
    sub  = real_meta[real_meta['IDPSJ'] == tid]['grade']
    plow = (sub <= 3).mean() * 100
    phi  = (sub >= 7).mean() * 100
    print(f"{tid:>6} {len(sub):>4} {r['y_test'].mean():>6.2f} {sub.std():>6.2f} "
          f"{plow:>5.1f}% {phi:>5.1f}% "
          f"| {r['y_pred'].mean():>7.2f} {r['mae']:>8.4f} {r['rmse']:>8.4f} {r['qwk']:>7.4f}")

# ── Plot: histogram aktual vs prediksi untuk tiap fold ────────────────────────────
sorted_r = sorted(fold_results, key=lambda x: x['test_idpsj'])
fig, axes = plt.subplots(4, 5, figsize=(22, 16))
axes = axes.flatten()

for i, r in enumerate(sorted_r):
    ax  = axes[i]
    tid = r['test_idpsj']
    bins = np.arange(0.5, 11.5, 1)
    ax.hist(r['y_test'], bins=bins, alpha=0.6, color='steelblue',  label='Aktual',   density=True)
    ax.hist(r['y_pred'], bins=bins, alpha=0.6, color='darkorange', label='Prediksi', density=True)
    ax.axvline(r['y_test'].mean(), color='blue',   linestyle='--', lw=1.5)
    ax.axvline(r['y_pred'].mean(), color='orange', linestyle='--', lw=1.5)
    ax.set_title(f"IDPSJ {tid}  MAE={r['mae']:.2f}  QWK={r['qwk']:.2f}\n"
                 f"act_\u03bc={r['y_test'].mean():.1f}  pred_\u03bc={r['y_pred'].mean():.1f}",
                 fontsize=7)
    ax.set_xticks(range(1, 11))
    ax.legend(fontsize=6)
    ax.set_xlabel('Grade'); ax.set_ylabel('Density')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribusi Grade Aktual vs Prediksi per IDPSJ \u2014 Siamese BiLSTM v11 (No Seed)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'diagnosis_dist_v11_no_seed.png'), dpi=100)
plt.show()